# 10 · Normalized Intensity Scores
Compute unpadded raw dB levels (first 4 s), extract empirical percentile bounds (p5, p95) from the UrbanSound8K training folds, normalize scores to [0, 1], and validate acoustic calibration.

In [1]:
# Cell 1 – interpreter setup & imports
import sys
from pathlib import Path
import time
import hashlib

print('sys.executable:', sys.executable)
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT     :', REPO_ROOT)

import numpy as np
import pandas as pd
import librosa
import joblib
from scipy.stats import mannwhitneyu
from tqdm import tqdm

from src.preprocessing.intensity import raw_db, raw_db_unpadded
from src.preprocessing.audio import SR, DURATION
print('Successfully imported raw_db and raw_db_unpadded from src.preprocessing.intensity')

sys.executable: C:\Users\LENOVO\miniconda3\envs\citypulse\python.exe
REPO_ROOT     : C:\Users\LENOVO\Desktop\CityPulse


Successfully imported raw_db and raw_db_unpadded from src.preprocessing.intensity


In [2]:
# Cell 2 – verify sha256 of inputs, load labels, V, and db_pad
def compute_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

labels_path = REPO_ROOT / 'data/processed/labels.csv'
features_path = REPO_ROOT / 'data/processed/audio_features.npy'
rawdb_path = REPO_ROOT / 'data/processed/audio_rawdb.npy'
valid_path = REPO_ROOT / 'data/processed/audio_rawdb_valid.npy'

labels_sha = compute_sha256(labels_path)
features_sha = compute_sha256(features_path)
rawdb_sha = compute_sha256(rawdb_path)
valid_sha = compute_sha256(valid_path)

print(f'labels.csv sha256            : {labels_sha}')
print(f'audio_features.npy sha256    : {features_sha}')
print(f'audio_rawdb.npy sha256       : {rawdb_sha}')
print(f'audio_rawdb_valid.npy sha256 : {valid_sha}')

EXPECTED_LABELS_SHA = '8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854'
EXPECTED_FEAT_SHA   = 'ed16a16f2a9e5c90aa61e925b461e0bce053c4654f1cc22a0812264630b1c73a'
EXPECTED_RAWDB_SHA  = '813337332360ac1587df32bea169807ab9d5380836b9d313285aad7e8740e27a'

assert labels_sha == EXPECTED_LABELS_SHA, f'labels.csv SHA mismatch: {labels_sha}'
assert features_sha == EXPECTED_FEAT_SHA, f'audio_features.npy SHA mismatch: {features_sha}'
assert rawdb_sha == EXPECTED_RAWDB_SHA, f'audio_rawdb.npy SHA mismatch: {rawdb_sha}'

labels = pd.read_csv(labels_path)
V = np.load(valid_path)
db_pad = np.load(rawdb_path)

assert len(labels) == 6558, f'Expected 6558 rows in labels, got {len(labels)}'
assert V.shape == (6558, 2), f'Expected V.shape == (6558, 2), got {V.shape}'
assert db_pad.shape == (6558,), f'Expected db_pad.shape == (6558,), got {db_pad.shape}'
print('Cell 2 SHA256 asserts and loads passed ✓')

labels.csv sha256            : 8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854
audio_features.npy sha256    : ed16a16f2a9e5c90aa61e925b461e0bce053c4654f1cc22a0812264630b1c73a
audio_rawdb.npy sha256       : 813337332360ac1587df32bea169807ab9d5380836b9d313285aad7e8740e27a
audio_rawdb_valid.npy sha256 : e5c5d7ca5924bdf29198112e16cca46ef5b7f5a653e2b6b4dd132a26a8d073b7
Cell 2 SHA256 asserts and loads passed ✓


In [3]:
# Cell 3 – compute db_final with raw_db_unpadded for every row and verify
n_clips = len(labels)
db_final_list = []
failures = []

t0 = time.perf_counter()
for idx, row in tqdm(labels.iterrows(), total=n_clips, desc='Computing unpadded raw dB', mininterval=5.0):
    rel_path = row['path']
    abs_path = REPO_ROOT / rel_path
    try:
        val = raw_db_unpadded(abs_path)
        db_final_list.append(val)
    except Exception as e:
        failures.append((idx, rel_path, str(e)))

elapsed = time.perf_counter() - t0
print(f'Total computation time: {elapsed:.2f} s ({elapsed/60:.2f} min)')
print(f'Number of failures    : {len(failures)}')

assert len(failures) == 0, f'Encountered {len(failures)} failures during processing'

db_final = np.array(db_final_list, dtype=np.float64)
assert db_final.shape == (6558,), f'Expected db_final.shape == (6558,), got {db_final.shape}'
assert not np.isnan(db_final).any(), 'NaN values found in db_final'
assert not np.isinf(db_final).any(), 'Inf values found in db_final'

n_valid = V[:, 0]
mask_le = (n_valid <= 88200)
mask_gt = (n_valid > 88200)

count_le = int(np.sum(mask_le))
count_gt = int(np.sum(mask_gt))

print(f'Size of group (n_valid <= 88200): {count_le} (expect 6357)')
print(f'Size of group (n_valid > 88200) : {count_gt} (expect 201)')

assert count_le == 6357, f'Expected 6357 rows with n_valid <= 88200, got {count_le}'
assert count_gt == 201, f'Expected 201 rows with n_valid > 88200, got {count_gt}'

assert np.allclose(db_final[mask_le], V[mask_le, 1], atol=1e-6), 'db_final does not match V[:, 1] on n_valid <= 88200'
assert np.allclose(db_final[mask_gt], db_pad[mask_gt], atol=1e-6), 'db_final does not match db_pad on n_valid > 88200'

print('Cell 3 asserts passed ✓')

Computing unpadded raw dB:   0%|          | 0/6558 [00:00<?, ?it/s]

Computing unpadded raw dB:   2%|▏         | 105/6558 [00:05<05:07, 20.97it/s]

Computing unpadded raw dB:  11%|█         | 701/6558 [00:10<01:14, 78.67it/s]

Computing unpadded raw dB:  24%|██▍       | 1559/6558 [00:15<00:41, 121.05it/s]

Computing unpadded raw dB:  33%|███▎      | 2165/6558 [00:21<00:39, 109.88it/s]

Computing unpadded raw dB:  42%|████▏     | 2722/6558 [00:26<00:35, 106.83it/s]

... (3 outputs left out)

Computing unpadded raw dB:  87%|████████▋ | 5716/6558 [00:46<00:05, 144.16it/s]

Computing unpadded raw dB: 100%|█████████▉| 6539/6558 [00:51<00:00, 150.36it/s]

Computing unpadded raw dB: 100%|██████████| 6558/6558 [00:51<00:00, 126.41it/s]

Total computation time: 51.89 s (0.86 min)
Number of failures    : 0
Size of group (n_valid <= 88200): 6357 (expect 6357)
Size of group (n_valid > 88200) : 201 (expect 201)
Cell 3 asserts passed ✓


In [4]:
# Cell 4 – compute p5 and p95 bounds on US8K training folds (1-9) and save
us8k_mask = (labels['source'] == 'us8k')
train_mask = us8k_mask & (labels['fold'] <= 9)
fold10_mask = us8k_mask & (labels['fold'] == 10)

n_train = int(np.sum(train_mask))
n_f10 = int(np.sum(fold10_mask))

print(f'US8K fold <= 9 (train) rows : {n_train} (expect 5753)')
print(f'US8K fold == 10 (test) rows  : {n_f10} (expect 605)')

assert n_train == 5753, f'Expected 5753 train rows, got {n_train}'
assert n_f10 == 605, f'Expected 605 fold 10 rows, got {n_f10}'

train_db = db_final[train_mask]
p5, p95 = np.percentile(train_db, [5, 95])
p5 = float(p5)
p95 = float(p95)

print(f'p5  (5th percentile) : {p5:.4f} dB')
print(f'p95 (95th percentile): {p95:.4f} dB')

assert p95 > p5, f'p95 ({p95}) must be strictly greater than p5 ({p5})'

bounds_meta = {
    'p5': p5,
    'p95': p95,
    'train_filter': 'source==us8k and fold<=9',
    'definition': 'RMS dB over unpadded samples, first 4 s'
}

models_dir = REPO_ROOT / 'models'
models_dir.mkdir(parents=True, exist_ok=True)
bounds_path = models_dir / 'intensity_bounds.joblib'
joblib.dump(bounds_meta, bounds_path)

print(f'Saved bounds dictionary to: {bounds_path}')
print('Cell 4 asserts passed ✓')

US8K fold <= 9 (train) rows : 5753 (expect 5753)
US8K fold == 10 (test) rows  : 605 (expect 605)
p5  (5th percentile) : -41.2172 dB
p95 (95th percentile): -13.8581 dB
Saved bounds dictionary to: C:\Users\LENOVO\Desktop\CityPulse\models\intensity_bounds.joblib
Cell 4 asserts passed ✓


In [5]:
# Cell 5 – compute normalized intensity scores [0, 1], save, and summarize
score = np.clip((db_final - p5) / (p95 - p5), 0.0, 1.0)

assert score.shape == (6558,), f'Expected score.shape == (6558,), got {score.shape}'
assert not np.isnan(score).any(), 'NaN values found in score'
assert not np.isinf(score).any(), 'Inf values found in score'
assert np.all((score >= 0.0) & (score <= 1.0)), 'Scores out of [0, 1] bounds found'

intensity_path = REPO_ROOT / 'data/processed/audio_intensity.npy'
np.save(intensity_path, score)
print(f'Saved audio_intensity.npy to: {intensity_path}')

esc50_mask = (labels['source'] == 'esc50')

groups = [
    ('us8k train folds 1-9', train_mask),
    ('us8k fold 10', fold10_mask),
    ('esc50', esc50_mask)
]

print('\n=== Group Intensity Score Summary ===')
for name, mask in groups:
    s = score[mask]
    cnt = len(s)
    mean_val = float(np.mean(s))
    median_val = float(np.median(s))
    frac_0 = float(np.mean(s == 0.0))
    frac_1 = float(np.mean(s == 1.0))
    
    print(f'{name:22s} | count: {cnt:4d} | mean: {mean_val:.4f} | median: {median_val:.4f} | frac==0: {frac_0:.4f} | frac==1: {frac_1:.4f}')
    
    if name == 'us8k train folds 1-9':
        assert abs(frac_0 - 0.05) <= 0.01, f'Training frac==0 ({frac_0}) is not within 0.01 of 0.05'
        assert abs(frac_1 - 0.05) <= 0.01, f'Training frac==1 ({frac_1}) is not within 0.01 of 0.05'

print('\nCell 5 asserts passed ✓')

Saved audio_intensity.npy to: C:\Users\LENOVO\Desktop\CityPulse\data\processed\audio_intensity.npy

=== Group Intensity Score Summary ===
us8k train folds 1-9   | count: 5753 | mean: 0.5901 | median: 0.6277 | frac==0: 0.0501 | frac==1: 0.0501
us8k fold 10           | count:  605 | mean: 0.5678 | median: 0.6458 | frac==0: 0.0612 | frac==1: 0.0413
esc50                  | count:  200 | mean: 0.7926 | median: 0.8429 | frac==0: 0.0000 | frac==1: 0.2300

Cell 5 asserts passed ✓


In [6]:
# Cell 6 – US8K per-class score summary, acoustic sanity checks, and AUC
df_us8k = labels[us8k_mask].copy()
df_us8k['score'] = score[us8k_mask]
df_us8k['db_final'] = db_final[us8k_mask]
df_us8k['db_pad'] = db_pad[us8k_mask]

class_summary = df_us8k.groupby('original_class')['score'].agg(['median', 'mean']).sort_values('median')
print('=== US8K: Per-original_class Score Summary (sorted by median) ===')
print(class_summary.round(4).to_string())

ac_med = float(class_summary.loc['air_conditioner', 'median'])
jh_med = float(class_summary.loc['jackhammer', 'median'])
ac_mean = float(class_summary.loc['air_conditioner', 'mean'])
jh_mean = float(class_summary.loc['jackhammer', 'mean'])

gap_med = jh_med - ac_med
gap_mean = jh_mean - ac_mean

print(f'\nair_conditioner median: {ac_med:.4f} | jackhammer median: {jh_med:.4f}')
print(f'Score gap (median): {gap_med:.4f}')
print(f'air_conditioner mean  : {ac_mean:.4f} | jackhammer mean  : {jh_mean:.4f}')
print(f'Score gap (mean)  : {gap_mean:.4f}')

assert ac_med < jh_med, f'median(air_conditioner) {ac_med} >= median(jackhammer) {jh_med}'
assert ac_mean < jh_mean, f'mean(air_conditioner) {ac_mean} >= mean(jackhammer) {jh_mean}'

# Mann-Whitney U / AUC calculation
jh_all = df_us8k[df_us8k['original_class'] == 'jackhammer']['db_final']
ac_all = df_us8k[df_us8k['original_class'] == 'air_conditioner']['db_final']
u_all = mannwhitneyu(jh_all, ac_all).statistic
auc_all = float(u_all / (len(jh_all) * len(ac_all)))

df_f10 = df_us8k[df_us8k['fold'] == 10]
jh_f10 = df_f10[df_f10['original_class'] == 'jackhammer']['db_final']
ac_f10 = df_f10[df_f10['original_class'] == 'air_conditioner']['db_final']
u_f10 = mannwhitneyu(jh_f10, ac_f10).statistic
auc_f10 = float(u_f10 / (len(jh_f10) * len(ac_f10)))

jh_pad = df_us8k[df_us8k['original_class'] == 'jackhammer']['db_pad']
ac_pad = df_us8k[df_us8k['original_class'] == 'air_conditioner']['db_pad']
u_pad = mannwhitneyu(jh_pad, ac_pad).statistic
auc_pad = float(u_pad / (len(jh_pad) * len(ac_pad)))

print(f'\nAUC (jackhammer vs air_conditioner) db_final [all US8K] : {auc_all:.4f}')
print(f'AUC (jackhammer vs air_conditioner) db_final [fold 10]   : {auc_f10:.4f}')
print(f'AUC (jackhammer vs air_conditioner) db_pad   [all US8K] : {auc_pad:.4f}')

assert auc_all > 0.5, f'Expected all-us8k db_final AUC > 0.5, got {auc_all}'
print('Cell 6 asserts passed ✓')

=== US8K: Per-original_class Score Summary (sorted by median) ===
                 median    mean
original_class                 
siren            0.5132  0.5082
air_conditioner  0.5741  0.5050
engine_idling    0.5824  0.6059
street_music     0.6219  0.5914
drilling         0.6597  0.6318
jackhammer       0.6747  0.6437
car_horn         0.7008  0.6724

air_conditioner median: 0.5741 | jackhammer median: 0.6747
Score gap (median): 0.1006
air_conditioner mean  : 0.5050 | jackhammer mean  : 0.6437
Score gap (mean)  : 0.1387

AUC (jackhammer vs air_conditioner) db_final [all US8K] : 0.6265
AUC (jackhammer vs air_conditioner) db_final [fold 10]   : 0.5932
AUC (jackhammer vs air_conditioner) db_pad   [all US8K] : 0.6031
Cell 6 asserts passed ✓


In [7]:
# Cell 7 – reload artifacts from disk, spot-check 5 random rows, and print SHA256 hashes
reloaded_score = np.load(intensity_path)
reloaded_bounds = joblib.load(bounds_path)

assert np.array_equal(reloaded_score, score), 'Reloaded audio_intensity.npy does not equal in-memory score exactly'
print('Reloaded audio_intensity.npy matches in-memory score exactly ✓')

b_p5 = reloaded_bounds['p5']
b_p95 = reloaded_bounds['p95']

sample_5 = labels.sample(5, random_state=7)
print(f'Spot check 5 random rows (random_state=7): {sample_5.index.tolist()}\n')

for idx in sample_5.index:
    row = labels.iloc[idx]
    path_i = REPO_ROOT / row['path']
    db_scratch = raw_db_unpadded(path_i)
    score_scratch = float(np.clip((db_scratch - b_p5) / (b_p95 - b_p5), 0.0, 1.0))
    score_saved = float(reloaded_score[idx])
    
    diff = abs(score_scratch - score_saved)
    print(f'Row {idx:4d} | app_class: {row["app_class"]:18s} | source: {row["source"]:5s} | scratch: {score_scratch:.8f} | saved: {score_saved:.8f} | diff: {diff:.2e}')
    assert diff < 1e-9, f'Row {idx} scratch score {score_scratch} != saved {score_saved}'

print('\nAll 5 spot checks passed ✓')

intensity_sha = compute_sha256(intensity_path)
bounds_sha = compute_sha256(bounds_path)

print(f'\naudio_intensity.npy     sha256: {intensity_sha}')
print(f'intensity_bounds.joblib sha256: {bounds_sha}')
print('Cell 7 asserts passed ✓')

Reloaded audio_intensity.npy matches in-memory score exactly ✓
Spot check 5 random rows (random_state=7): [896, 5779, 351, 1175, 1188]

Row  896 | app_class: construction       | source: us8k  | scratch: 0.66163166 | saved: 0.66163166 | diff: 0.00e+00
Row 5779 | app_class: construction       | source: us8k  | scratch: 0.72776215 | saved: 0.72776215 | diff: 0.00e+00
Row  351 | app_class: construction       | source: us8k  | scratch: 0.42718310 | saved: 0.42718310 | diff: 0.00e+00
Row 1175 | app_class: music_loudspeaker  | source: us8k  | scratch: 0.75971899 | saved: 0.75971899 | diff: 0.00e+00
Row 1188 | app_class: music_loudspeaker  | source: us8k  | scratch: 0.87173788 | saved: 0.87173788 | diff: 0.00e+00

All 5 spot checks passed ✓

audio_intensity.npy     sha256: 1277608d8edcc60b5d5bb85a2f04988600c5be3264dc152b0ba8edc4de33f811
intensity_bounds.joblib sha256: 22b0cdb61eb19769edb41f1e88049f7625e2b9b089cec9788f165ff8e4155340
Cell 7 asserts passed ✓
